# Notebook 04 — Constraint Phase Maps

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 01 mapped input distributions.  
Notebook 02 mapped cache and branching proxy structure.  
Notebook 03 mapped scalar vs SIMD execution-path suitability.  

Notebook 04 combines those layers into **constraint phase maps**.

Constraint view:
> performance regimes emerge where distribution structure, branch/cache behavior, and execution paths align.

## Goals

1. Load outputs from Notebook 01–03 when available.
2. Merge metrics by distribution name.
3. Build phase-map coordinates:
   - entropy / repetition
   - branch pressure / locality
   - scalar / SIMD suitability
   - coherence / fragmentation proxy
4. Produce figures:
   - constraint phase map
   - coherence landscape
   - regime classification map
   - RML summary radar-style table
5. Export CSV, JSON, Markdown report, and PNG outputs.

This notebook still uses structural proxies. Later notebooks can overlay real benchmark throughput, latency, hardware counters, and upstream benchmark outputs.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load prior notebook outputs

This notebook expects:

- `notebook01_input_distribution_metrics.csv`
- `notebook02_cache_branching_metrics.csv`
- `notebook03_simd_scalar_path_metrics.csv`

If those are missing, fallback tables are used so the notebook remains runnable.

In [ ]:
def load_csv_or_none(path):
    if path.exists():
        print("Loaded:", path)
        return pd.read_csv(path)
    print("Missing:", path)
    return None

df01 = load_csv_or_none(RESULTS_DIR / "notebook01_input_distribution_metrics.csv")
df02 = load_csv_or_none(RESULTS_DIR / "notebook02_cache_branching_metrics.csv")
df03 = load_csv_or_none(RESULTS_DIR / "notebook03_simd_scalar_path_metrics.csv")

if df01 is None:
    df01 = pd.DataFrame([
        {"name": "low_entropy_repeating", "approx_entropy_bits": 2.0, "repetition_ratio": 0.99996, "unique_count": 4, "delta_abs_mean": 1.5},
        {"name": "sequential_ids", "approx_entropy_bits": 9.0, "repetition_ratio": 0.0, "unique_count": 100000, "delta_abs_mean": 1.0},
        {"name": "uniform_32bit", "approx_entropy_bits": 9.0, "repetition_ratio": 0.00001, "unique_count": 99999, "delta_abs_mean": 1.4e9},
        {"name": "zipfian_smallints", "approx_entropy_bits": 0.01, "repetition_ratio": 0.80552, "unique_count": 19448, "delta_abs_mean": 7.0e7},
        {"name": "clustered_ranges", "approx_entropy_bits": 3.39, "repetition_ratio": 0.98, "unique_count": 2000, "delta_abs_mean": 5.0e4},
    ])

if df02 is None:
    df02 = pd.DataFrame([
        {"name": "low_entropy_repeating", "branch_pressure_score": 0.01, "digit_length_entropy": 0.0, "locality_small_delta_ratio": 1.0, "cache_window_reuse_proxy": 0.94},
        {"name": "sequential_ids", "branch_pressure_score": 0.20, "digit_length_entropy": 0.52, "locality_small_delta_ratio": 1.0, "cache_window_reuse_proxy": 0.0},
        {"name": "uniform_32bit", "branch_pressure_score": 0.72, "digit_length_entropy": 0.90, "locality_small_delta_ratio": 0.0, "cache_window_reuse_proxy": 0.0},
        {"name": "zipfian_smallints", "branch_pressure_score": 0.72, "digit_length_entropy": 2.42, "locality_small_delta_ratio": 0.27, "cache_window_reuse_proxy": 0.35},
        {"name": "clustered_ranges", "branch_pressure_score": 0.79, "digit_length_entropy": 1.25, "locality_small_delta_ratio": 0.0, "cache_window_reuse_proxy": 0.01},
    ])

if df03 is None:
    df03 = pd.DataFrame([
        {"name": "low_entropy_repeating", "simd_suitability": 0.26, "scalar_suitability": 0.98, "estimated_speedup_simd_over_scalar": 0.82},
        {"name": "sequential_ids", "simd_suitability": 0.39, "scalar_suitability": 0.62, "estimated_speedup_simd_over_scalar": 1.08},
        {"name": "uniform_32bit", "simd_suitability": 0.62, "scalar_suitability": 0.08, "estimated_speedup_simd_over_scalar": 1.45},
        {"name": "zipfian_smallints", "simd_suitability": 0.67, "scalar_suitability": 0.22, "estimated_speedup_simd_over_scalar": 1.37},
        {"name": "clustered_ranges", "simd_suitability": 0.45, "scalar_suitability": 0.05, "estimated_speedup_simd_over_scalar": 1.20},
    ])

# Merge, avoiding duplicate overlapping columns.
base = df01.copy()
merged = base.merge(df02, on="name", how="outer", suffixes=("", "_n2"))
merged = merged.merge(df03, on="name", how="outer", suffixes=("", "_n3"))

merged

## Build phase-map metrics

The phase-map layer turns multiple proxy metrics into interpretable coordinates:

- **structure_regularization**: locality + reuse + low branch pressure.
- **execution_alignment**: scalar/SIMD balance and speedup stability.
- **coherence_score**: stable structure across distribution + execution constraints.
- **fragmentation_score**: branch pressure + low locality + low reuse.

In [ ]:
work = merged.copy()

def norm01(series):
    s = pd.Series(series).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

# Fill missing with conservative defaults.
for col in [
    "approx_entropy_bits", "repetition_ratio", "delta_abs_mean",
    "branch_pressure_score", "locality_small_delta_ratio",
    "cache_window_reuse_proxy", "simd_suitability", "scalar_suitability",
    "estimated_speedup_simd_over_scalar"
]:
    if col not in work.columns:
        work[col] = np.nan
    work[col] = work[col].fillna(work[col].median())

work["entropy_norm"] = norm01(work["approx_entropy_bits"])
work["delta_norm"] = norm01(np.log10(work["delta_abs_mean"].astype(float) + 1.0))
work["branch_norm"] = norm01(work["branch_pressure_score"])
work["speedup_norm"] = norm01(work["estimated_speedup_simd_over_scalar"])

work["structure_regularization"] = (
    0.35 * work["locality_small_delta_ratio"] +
    0.30 * work["cache_window_reuse_proxy"] +
    0.20 * (1.0 - work["branch_norm"]) +
    0.15 * work["repetition_ratio"]
).clip(0, 1)

work["execution_alignment"] = (
    0.40 * work["speedup_norm"] +
    0.30 * work["simd_suitability"] +
    0.20 * work["scalar_suitability"] +
    0.10 * (1.0 - np.abs(work["simd_suitability"] - work["scalar_suitability"]))
).clip(0, 1)

work["fragmentation_score"] = (
    0.45 * work["branch_norm"] +
    0.30 * (1.0 - work["locality_small_delta_ratio"]) +
    0.25 * (1.0 - work["cache_window_reuse_proxy"])
).clip(0, 1)

work["coherence_score"] = (
    0.45 * work["structure_regularization"] +
    0.35 * work["execution_alignment"] +
    0.20 * (1.0 - work["fragmentation_score"])
).clip(0, 1)

def classify(row):
    if row["coherence_score"] >= 0.65 and row["structure_regularization"] >= 0.55:
        return "coherent-local"
    if row["simd_suitability"] >= 0.55 and row["estimated_speedup_simd_over_scalar"] > 1.2:
        return "simd-favorable"
    if row["fragmentation_score"] >= 0.70:
        return "fragmented-irregular"
    if row["scalar_suitability"] >= 0.55:
        return "scalar-favorable"
    return "mixed-transition"

work["regime"] = work.apply(classify, axis=1)

phase_cols = [
    "name", "regime", "coherence_score", "fragmentation_score",
    "structure_regularization", "execution_alignment",
    "approx_entropy_bits", "repetition_ratio",
    "branch_pressure_score", "simd_suitability", "scalar_suitability",
    "estimated_speedup_simd_over_scalar"
]

work[phase_cols].sort_values("coherence_score", ascending=False)

## Export phase-map table

In [ ]:
csv_path = RESULTS_DIR / "notebook04_constraint_phase_metrics.csv"
json_path = RESULTS_DIR / "notebook04_constraint_phase_metrics.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Constraint phase map

X-axis: fragmentation pressure.  
Y-axis: coherence score.  
Points are distributions/regimes.

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook04_constraint_phase_map.png"

plt.figure(figsize=(8, 6))
plt.scatter(work["fragmentation_score"], work["coherence_score"])
for _, row in work.iterrows():
    plt.annotate(row["name"], (row["fragmentation_score"], row["coherence_score"]), fontsize=8)
plt.xlabel("Fragmentation score")
plt.ylabel("Coherence score")
plt.title("Constraint Phase Map: Coherence vs Fragmentation")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Coherence landscape

This figure combines structure regularization and execution alignment.

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook04_coherence_landscape.png"

plt.figure(figsize=(8, 6))
sizes = 120 + 500 * work["coherence_score"]
plt.scatter(work["structure_regularization"], work["execution_alignment"], s=sizes, alpha=0.75)
for _, row in work.iterrows():
    plt.annotate(row["name"], (row["structure_regularization"], row["execution_alignment"]), fontsize=8)
plt.xlabel("Structure regularization")
plt.ylabel("Execution alignment")
plt.title("Coherence Landscape: Structure × Execution")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Regime classification map

This bar chart ranks distributions by coherence and labels each regime.

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook04_regime_classification.png"

plot_df = work.sort_values("coherence_score")
plt.figure(figsize=(10, 5))
bars = plt.bar(plot_df["name"], plot_df["coherence_score"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Coherence score")
plt.title("Regime Classification by Coherence Score")

for i, (_, row) in enumerate(plot_df.iterrows()):
    plt.text(i, row["coherence_score"] + 0.02, row["regime"], rotation=90, ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — RML constraint summary matrix

A compact heatmap-style matrix showing normalized metrics by distribution.

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook04_constraint_summary_matrix.png"

matrix_cols = [
    "entropy_norm",
    "repetition_ratio",
    "branch_norm",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "simd_suitability",
    "scalar_suitability",
    "coherence_score",
]

mat = work.set_index("name")[matrix_cols].sort_values("coherence_score", ascending=False)

plt.figure(figsize=(10, 5))
plt.imshow(mat.values, aspect="auto")
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(matrix_cols)), matrix_cols, rotation=45, ha="right")
plt.colorbar(label="Normalized score")
plt.title("RML Constraint Summary Matrix")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_04_constraint_phase_maps.md"

summary_cols = [
    "name", "regime", "coherence_score", "fragmentation_score",
    "structure_regularization", "execution_alignment",
    "branch_pressure_score", "simd_suitability", "scalar_suitability",
]

lines = [
    "# Report 04 — Constraint Phase Maps",
    "",
    "This report combines distribution, cache/branching, and execution-path metrics into an RML-style phase-map layer.",
    "",
    "Constraint view:",
    "> performance regimes emerge where distribution structure, branch/cache behavior, and execution paths align.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    "",
    "## Constraint phase summary",
    "",
    work[summary_cols].sort_values("coherence_score", ascending=False).to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Coherence score combines structure regularization, execution alignment, and low fragmentation.",
    "- Fragmentation score highlights branch pressure, weak locality, and weak local reuse.",
    "- SIMD-favorable regimes differ from scalar-favorable regimes; neither is universally best.",
    "- Phase maps create a bridge from synthetic distributions to real benchmark performance.",
    "",
    "## Next step",
    "",
    "Notebook 05 should integrate real benchmark outputs from upstream runs and compare observed throughput against these structural predictions.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook04_constraint_phase_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook04_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_04_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))